In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Finetune

In [ ]:
pip install sentence-transformers

In [ ]:
pip install timm

In [ ]:
#!pip install git+https://github.com/huggingface/accelerate

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
from scipy import spatial
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
import timm
from timm.utils import AverageMeter
import sys
sys.path.append('/kaggle/input/backbone/sentence-transformers/sentence-transformers')
from sentence_transformers import SentenceTransformer
##import accelerate
#from accelerate import Accelerator
#from accelerate.utils import set_seed

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 2epoch: trn/loss=0.2439, trn/cos=0.7561 | val/loss=0.2226, val/cos=0.7774
class CFG:
    #model_name = 'vit_huge_patch14_224'
    #model_path = '/kaggle/input/swin-large-finetune-stablediffusion-textimage-pair/swin_large_patch4_window7_224_15_epochs.pth'
    #model_name = 'swin_large_patch4_window7_224'
    #input_size = 224
    model_path = '/kaggle/input/jing-backbone/test_sd2_only_4_epoch.pth'
    model_name = 'vit_large_patch16_384'
    save_name = 'vit_large_patch16_384_sd2_only_5_epoch'
    input_size = 384
    batch_size = 32
    num_epochs = 1
    #mixed_precision = 'no'
    lr = 1e-4
    seed = 2023

In [ ]:
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True


seed_everything(CFG.seed)

In [ ]:
class DiffusionDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['filepath'])
        image = self.transform(image)
        prompt = row['prompt']
        return image, prompt

In [ ]:
class DiffusionCollator:
    def __init__(self):
        self.st_model = SentenceTransformer(
            '/kaggle/input/sentence-transformers-222/all-MiniLM-L6-v2',
            device='cpu'
        )
    
    def __call__(self, batch):
        images, prompts = zip(*batch)
        images = torch.stack(images)
        prompt_embeddings = self.st_model.encode(
            prompts, 
            show_progress_bar=False, 
            convert_to_tensor=True
        )
        return images, prompt_embeddings

In [ ]:
def get_dataloaders(
    trn_df,
    val_df,
    input_size,
    batch_size
):
    transform = transforms.Compose([
        transforms.Resize(input_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    trn_dataset = DiffusionDataset(trn_df, transform)
    val_dataset = DiffusionDataset(val_df, transform)
    collator = DiffusionCollator()
    
    dataloaders = {}
    dataloaders['train'] = DataLoader(
        dataset=trn_dataset,
        shuffle=True,
        batch_size=batch_size,
        pin_memory=True,
        num_workers=2,
        drop_last=True,
        collate_fn=collator
    )
    dataloaders['val'] = DataLoader(
        dataset=val_dataset,
        shuffle=False,
        batch_size=batch_size,
        pin_memory=True,
        num_workers=2,
        drop_last=False,
        collate_fn=collator
    )
    return dataloaders


In [ ]:
def cosine_similarity(y_trues, y_preds):
    return np.mean([
        1 - spatial.distance.cosine(y_true, y_pred) 
        for y_true, y_pred in zip(y_trues, y_preds)
    ])

In [ ]:
def train(
    trn_df,
    val_df,
    model_path,
    model_name,
    input_size,
    batch_size,
    num_epochs,
    lr,
    save_name
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dataloaders = get_dataloaders(
        trn_df,
        val_df,
        input_size,
        batch_size
    )

    """
    model = timm.create_model(
    CFG.model_name,
    pretrained=True,
    num_classes=384
    )
    model.set_grad_checkpointing()
    model.to(device)
    """
    model = timm.create_model(
    CFG.model_name,
    pretrained=False,
    num_classes=384
    )
    
    state_dict = torch.load(model_path)
    model.load_state_dict(state_dict)
    model.set_grad_checkpointing()
    model.to(device)
    model.eval()
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    ttl_iters = num_epochs * len(dataloaders['train'])
    scheduler = CosineAnnealingLR(optimizer, T_max=ttl_iters, eta_min=1e-6)
    criterion = nn.CosineEmbeddingLoss()
    best_score = -1.0

    for epoch in range(num_epochs):
        train_meters = {
            'loss': AverageMeter(),
            'cos': AverageMeter(),
        }
        model.train()
        for X, y in tqdm(dataloaders['train'], leave=False):
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            X_out = model(X)
            target = torch.ones(X.size(0)).to(device)
            loss = criterion(X_out, y, target)
            loss.backward()

            optimizer.step()
            scheduler.step()

            trn_loss = loss.item()
            trn_cos = cosine_similarity(
                X_out.detach().cpu().numpy(), 
                y.detach().cpu().numpy()
            )

            train_meters['loss'].update(trn_loss, n=X.size(0))
            train_meters['cos'].update(trn_cos, n=X.size(0))

        print('Epoch {:d} / trn/loss={:.4f}, trn/cos={:.4f}'.format(
            epoch + 1,
            train_meters['loss'].avg,
            train_meters['cos'].avg))

        val_meters = {
            'loss': AverageMeter(),
            'cos': AverageMeter(),
        }
        model.eval()
        for X, y in tqdm(dataloaders['val'], leave=False):
            X, y = X.to(device), y.to(device)

            with torch.no_grad():
                X_out = model(X)
                target = torch.ones(X.size(0)).to(device)
                loss = criterion(X_out, y, target)

                val_loss = loss.item()
                val_cos = cosine_similarity(
                    X_out.detach().cpu().numpy(), 
                    y.detach().cpu().numpy()
                )

            val_meters['loss'].update(val_loss, n=X.size(0))
            val_meters['cos'].update(val_cos, n=X.size(0))

        print('Epoch {:d} / val/loss={:.4f}, val/cos={:.4f}'.format(
            epoch + 1,
            val_meters['loss'].avg,
            val_meters['cos'].avg))
        if val_meters['cos'].avg > best_score:
            best_score = val_meters['cos'].avg
            torch.save(model.state_dict(), f'{save_name}.pth')

In [ ]:
#df = pd.read_csv('/kaggle/input/diffusiondb-data-cleansing/diffusiondb.csv')

train_v1 = pd.read_csv('/kaggle/input/gustavosta-stable-diffusion-prompts-sd2/train.csv')
test_v1 = pd.read_csv('/kaggle/input/gustavosta-stable-diffusion-prompts-sd2/eval.csv')

train_v2 = pd.read_csv('/kaggle/input/gustavosta-stable-diffusion-prompts-sd2-v2/train.csv')
test_v2 = pd.read_csv('/kaggle/input/gustavosta-stable-diffusion-prompts-sd2-v2/eval.csv')

df = pd.DataFrame({"filepath":[], "prompt":[]})
df = df.append(pd.DataFrame({"filepath":train_v1['image_path'].values, "prompt":train_v1['Prompt'].values}))
df = df.append(pd.DataFrame({"filepath":test_v1['image_path'].values, "prompt":test_v1['Prompt'].values}))
df = df.append(pd.DataFrame({"filepath":train_v2['image_path'].values, "prompt":train_v2['Prompt'].values}))
df = df.append(pd.DataFrame({"filepath":test_v1['image_path'].values, "prompt":test_v1['Prompt'].values}))
df['filepath'] = df['filepath'].apply(lambda x : x.lower())

trn_df, val_df = train_test_split(df, test_size=0.1, random_state=CFG.seed)

In [ ]:
train(trn_df, val_df,CFG.model_path, CFG.model_name, CFG.input_size, CFG.batch_size, CFG.num_epochs, CFG.lr, CFG.save_name)